# Case Study 1 — Hospital Readmission Prediction (Logistic Regression, L2)

## Step 1 — Imports

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report


## Step 2 — Load the data

In [ ]:
df = pd.read_csv("dataset.csv")

print(df.shape)
df.head()


## Step 3 — Look at the data

In [ ]:
print(df.columns.tolist())
df.info()
print(df.describe())
print(df.isnull().sum())
print(df["target"].value_counts())
print(df["target"].value_counts(normalize=True))


## Step 4 — Split into train and test

In [ ]:
train, test = train_test_split(df, test_size=0.2, random_state=42, stratify=df["target"])
train = train.reset_index(drop=True)
test = test.reset_index(drop=True)

y_test_true = test["target"].copy()
test = test.drop(columns=["target"])

print(train.shape)
print(test.shape)


## Step 5 — Fill the missing values

In [ ]:
numeric_cols = train.drop(columns=["target"]).select_dtypes(include=np.number).columns
text_cols = train.select_dtypes(exclude=np.number).columns

for col in numeric_cols:
    train[col] = train[col].fillna(train[col].median())
    if col in test.columns:
        test[col] = test[col].fillna(train[col].median())

for col in text_cols:
    train[col] = train[col].fillna(train[col].mode()[0])
    if col in test.columns:
        test[col] = test[col].fillna(train[col].mode()[0])

print(train.isnull().sum().sum())
print(test.isnull().sum().sum())


## Step 6 — Turn text columns into numbers

In [ ]:
X = train.drop(columns=["target", "patient_id"])
y = train["target"]
X_test = test.drop(columns=["patient_id"], errors="ignore")

combined = pd.concat([X, X_test], keys=["train", "test"])
combined = pd.get_dummies(combined, drop_first=True)

X = combined.loc["train"]
X_test = combined.loc["test"]

print(X.shape)
print(X_test.shape)
X.head()


## Step 7 — Split train into train and validation

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(len(X_train))
print(len(X_valid))


## Step 8 — Scale the features

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid)


## Step 9 — Logistic Regression with L2 regularization

In [ ]:
model = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
model.fit(X_train_scaled, y_train)

pred_proba = model.predict_proba(X_valid_scaled)[:, 1]
pred_class = model.predict(X_valid_scaled)


## Step 10 — Evaluate with ROC-AUC and the confusion matrix

In [ ]:
auc = roc_auc_score(y_valid, pred_proba)
print("ROC-AUC:", auc)

cm = confusion_matrix(y_valid, pred_class)
print(cm)

tn, fp, fn, tp = cm.ravel()
print("True Negatives:", tn)
print("False Positives:", fp)
print("False Negatives:", fn)
print("True Positives:", tp)

print(classification_report(y_valid, pred_class))


## Step 11 — Train the final model on ALL the training data and predict test

In [ ]:
final_model = LogisticRegression(C=1.0, max_iter=1000, random_state=42)

scaler_final = StandardScaler()
X_scaled_full = scaler_final.fit_transform(X)
X_test_scaled_final = scaler_final.transform(X_test)

final_model.fit(X_scaled_full, y)

test_predictions_proba = final_model.predict_proba(X_test_scaled_final)[:, 1]
test_predictions_class = final_model.predict(X_test_scaled_final)

print(test_predictions_proba[:10])
print(test_predictions_class[:10])


## Step 12 — Save submission.csv

In [ ]:
submission = pd.DataFrame({
    "patient_id": test["patient_id"],
    "readmission_probability": test_predictions_proba,
    "predicted_readmitted_30d": test_predictions_class
})

submission.to_csv("submission.csv", index=False)
print(submission.shape)
submission.head()


## Step 13 — Bonus: check against the true holdout labels

In [ ]:
holdout_auc = roc_auc_score(y_test_true, test_predictions_proba)
print("Holdout ROC-AUC:", holdout_auc)
print(confusion_matrix(y_test_true, test_predictions_class))
